# Broiler Energy Partitioning Analysis
**MSc Research Data | University of Ibadan, Nigeria**

This notebook analyses energy partitioning indices in pre-starter broiler chicks (days 1-10) across three diet groups: Cereal, Protein, and Fibre sources. Two methods are compared: the Comparative Slaughter Technique (CST) and a nonlinear mathematical model.

**Author:** Shina James Owolabi, RAS | PhD Candidate, University of Ibadan  
**Supervisor:** Dr. Oluwafunmilayo O. Adeleye

---
## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120
})

COLORS = {
    'Cereal':  '#2E86AB',
    'Protein': '#A23B72',
    'Fibre':   '#F18F01',
    'CST':     '#2E86AB',
    'Model':   '#E84855'
}

In [ ]:
# Load all sheets
FILE = 'Energy_Partitioning_Corrected.xlsx'
xl = pd.read_excel(FILE, sheet_name=None, header=None)
print('Sheets loaded:', list(xl.keys()))

---
## 2. Raw Data Cleaning

In [ ]:
# Raw data: header is row index 1, data starts at row 2
raw = xl['1. Raw Data'].copy()
raw.columns = raw.iloc[1]
raw = raw.iloc[2:].reset_index(drop=True)
raw.columns = [
    'Rep_Code', 'Diet_Name', 'Group',
    'GE_Feed_kcal_kg', 'GE_Excreta_kcal_kg', 'Ti_Feed_pct', 'Ti_Excreta_pct',
    'ADFI_g_day', 'CFI_g_bird', 'BWI_g', 'BEF_g',
    'Init_GE_Carc_kcal_kg', 'Final_GE_Carc_kcal_kg',
    'Carcass_Fat_g', 'Carcass_Protein_g', 'n'
]
raw = raw.dropna(subset=['Rep_Code'])
for col in raw.columns[3:]:
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

print(f'Raw data: {raw.shape[0]} replicate observations')
print(f'Diet groups: {raw["Group"].unique()}')
raw.head(5)

In [ ]:
# Corrected calculations sheet
calc = xl['2. Corrected Calculations'].copy()
calc.columns = calc.iloc[1]
calc = calc.iloc[2:].reset_index(drop=True)
calc = calc[calc.iloc[:, 0].notna()]
calc = calc[~calc.iloc[:, 0].astype(str).str.startswith('REF')]
calc.columns = [
    'Rep_Code', 'Diet_Name', 'Group',
    'ME_kcal_kg', 'MEI_kcal', 'RE_kcal', 'HP_kcal',
    'REF_kcal', 'REP_kcal', 'KREF', 'KREP',
    'Fat_pct', 'Protein_pct', 'BWG_g', 'BWI_g', 'BEF_g',
    'CFI_g', 'ADFI_g_d', 'Init_GE_Carc', 'Final_GE_Carc'
]
for col in calc.columns[3:]:
    calc[col] = pd.to_numeric(calc[col], errors='coerce')
calc = calc.dropna(subset=['MEI_kcal'])

print(f'Calculated data: {calc.shape[0]} observations')
calc[['Rep_Code','Diet_Name','Group','MEI_kcal','RE_kcal','HP_kcal','KREF','KREP']].head(8)

---
## 3. Treatment Means

In [ ]:
# Treatment means
means_raw = xl['3. Treatment Means (T4.6-4.8)'].copy()
means_raw.columns = means_raw.iloc[1]
means_raw = means_raw.iloc[2:].reset_index(drop=True)
means_raw.columns = [
    'Group', 'Diet', 'n',
    'MEI_mean', 'MEI_sd', 'RE_mean', 'RE_sd',
    'HP_mean', 'HP_sd', 'REF_mean', 'REF_sd',
    'REP_mean', 'REP_sd', 'KREF_mean', 'KREF_sd', 'KREP_mean'
]
means = means_raw.dropna(subset=['Diet']).copy()
means = means[means['Diet'].astype(str).str.strip() != 'nan']
for col in means.columns[2:]:
    means[col] = pd.to_numeric(means[col], errors='coerce')

# Tag group by position
group_map = {
    'Reference': 'Cereal', 'Maize': 'Cereal', 'Pearl Millet': 'Cereal',
    'Sorghum': 'Cereal', 'Wheat': 'Cereal',
    'Full-Fat Soya': 'Protein', 'Groundnut Cake': 'Protein', 'Soya Bean Meal': 'Protein',
    'Brewers Dried Grain': 'Fibre', 'Wheat Bran': 'Fibre',
    'Rice Bran': 'Fibre', 'Corn Bran': 'Fibre'
}
means['Group_clean'] = means['Diet'].map(group_map).fillna('Reference')
means[['Group_clean', 'Diet', 'MEI_mean', 'RE_mean', 'HP_mean', 'KREF_mean', 'KREP_mean']]

---
## 4. Visualisation: Energy Fractions by Diet Group

Each bar shows mean MEI partitioned into HP (heat loss) and RE (retained energy) per bird over 10 days.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
groups = ['Cereal', 'Protein', 'Fibre']

cereal_diets  = ['Reference', 'Maize', 'Sorghum', 'Pearl Millet', 'Wheat']
protein_diets = ['Reference', 'Full-Fat Soya', 'Groundnut Cake', 'Soya Bean Meal']
fibre_diets   = ['Reference', 'Brewers Dried Grain', 'Wheat Bran', 'Rice Bran', 'Corn Bran']
group_diets   = [cereal_diets, protein_diets, fibre_diets]

for ax, group, diets, color in zip(axes, groups, group_diets, COLORS.values()):
    subset = means[means['Diet'].isin(diets)].set_index('Diet').reindex(diets)
    x = np.arange(len(diets))
    w = 0.35
    ax.bar(x - w/2, subset['HP_mean'], w, label='Heat Production (HP)', color=color, alpha=0.85)
    ax.bar(x + w/2, subset['RE_mean'], w, label='Retained Energy (RE)', color=color, alpha=0.45)
    ax.set_title(f'{group} Diets', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(diets, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Energy (kcal/bird/10d)')
    ax.legend(fontsize=8)

fig.suptitle('Heat Production vs Retained Energy by Diet (Pre-starter Broilers, Days 1-10)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig1_hp_vs_re_by_diet.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 1 saved.')

---
## 5. MEI Comparison Across All Diets

In [ ]:
all_diets_order = cereal_diets[1:] + protein_diets[1:] + fibre_diets[1:]
all_groups_color = (['#2E86AB'] * 4 + ['#A23B72'] * 3 + ['#F18F01'] * 4)

plot_data = means[means['Diet'].isin(all_diets_order)].set_index('Diet').reindex(all_diets_order)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(all_diets_order)), plot_data['MEI_mean'],
              color=all_groups_color, edgecolor='white', linewidth=0.5)

ax.axhline(y=plot_data['MEI_mean'].mean(), color='#333', linestyle='--',
           linewidth=1.2, label=f'Overall mean: {plot_data["MEI_mean"].mean():.1f} kcal')
ax.set_xticks(range(len(all_diets_order)))
ax.set_xticklabels(all_diets_order, rotation=35, ha='right')
ax.set_ylabel('MEI (kcal/bird/10d)')
ax.set_title('Metabolisable Energy Intake (MEI) Across All Experimental Diets', fontweight='bold')

patches = [
    mpatches.Patch(color='#2E86AB', label='Cereal'),
    mpatches.Patch(color='#A23B72', label='Protein'),
    mpatches.Patch(color='#F18F01', label='Fibre')
]
ax.legend(handles=patches + [plt.Line2D([0],[0], color='#333', linestyle='--', label=f'Mean {plot_data["MEI_mean"].mean():.0f} kcal')])

plt.tight_layout()
plt.savefig('fig2_mei_all_diets.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 2 saved.')

---
## 6. Fat vs Protein Energy Retention (REF vs REP)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for diet, row in means.set_index('Diet').iterrows():
    if pd.notna(row['REF_mean']) and pd.notna(row['REP_mean']):
        grp = group_map.get(diet, 'Reference')
        color = COLORS.get(grp, '#888')
        ax.scatter(row['REF_mean'], row['REP_mean'], color=color, s=80, zorder=3)
        ax.annotate(diet, (row['REF_mean'], row['REP_mean']),
                    textcoords='offset points', xytext=(5, 3), fontsize=8)

ax.set_xlabel('Retained Energy as Fat (REF, kcal/bird)')
ax.set_ylabel('Retained Energy as Protein (REP, kcal/bird)')
ax.set_title('Fat vs Protein Energy Retention by Diet', fontweight='bold')

patches = [mpatches.Patch(color=v, label=k) for k, v in COLORS.items() if k in ['Cereal','Protein','Fibre']]
ax.legend(handles=patches)
plt.tight_layout()
plt.savefig('fig3_ref_vs_rep.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 3 saved.')

---
## 7. Efficiency Indices: KREF and KREP

In [ ]:
eff_data = means[means['Diet'].isin(all_diets_order)].set_index('Diet').reindex(all_diets_order)

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(all_diets_order))
w = 0.35

ax.bar(x - w/2, eff_data['KREF_mean'], w, label='KREF (fat retention efficiency)',
       color='#2E86AB', alpha=0.85)
ax.bar(x + w/2, eff_data['KREP_mean'], w, label='KREP (protein retention efficiency)',
       color='#F18F01', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(all_diets_order, rotation=35, ha='right')
ax.set_ylabel('Efficiency Ratio (kcal retained / kcal MEI)')
ax.set_title('Efficiency of ME Use for Fat (KREF) and Protein (KREP) Retention', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('fig4_kref_krep.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 4 saved.')

---
## 8. CST vs Mathematical Model Comparison

In [ ]:
# Load CST vs Model sheet
cst_raw = xl['4. CST vs Model (T4.10-4.12)'].copy()
cst_raw.columns = cst_raw.iloc[5]
cst_raw = cst_raw.iloc[6:].reset_index(drop=True)
cst_raw.columns = [
    'Group', 'Diet', 'Method', 'BW_mean_kg', 'ADWG_mean_g_day',
    'MEId_kcal_10d', 'HP_kcal_10d', 'Delta_HP_pct',
    'RE_kcal_10d', 'Delta_RE_pct', 'NEg_kcal_10d', 'extra'
]
cst_raw = cst_raw.dropna(subset=['Method'])
cst_raw['Method'] = cst_raw['Method'].astype(str).str.strip()
for col in ['HP_kcal_10d', 'RE_kcal_10d', 'MEId_kcal_10d']:
    cst_raw[col] = pd.to_numeric(cst_raw[col], errors='coerce')

# Fill diet names forward
cst_raw['Diet'] = cst_raw['Diet'].astype(str).replace('nan', np.nan).ffill()
cst_raw = cst_raw[cst_raw['Method'].isin(['CST', 'Model'])]

hp_cst   = cst_raw[cst_raw['Method'] == 'CST']['HP_kcal_10d'].dropna().values
hp_model = cst_raw[cst_raw['Method'] == 'Model']['HP_kcal_10d'].dropna().values
diets_cst = cst_raw[cst_raw['Method'] == 'CST']['Diet'].values[:len(hp_cst)]

fig, ax = plt.subplots(figsize=(10, 6))
n = min(len(hp_cst), len(hp_model))
x = np.arange(n)
w = 0.35
ax.bar(x - w/2, hp_cst[:n],   w, label='CST',   color=COLORS['CST'],   alpha=0.85)
ax.bar(x + w/2, hp_model[:n], w, label='Model', color=COLORS['Model'], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(diets_cst[:n], rotation=35, ha='right')
ax.set_ylabel('Heat Production (kcal/bird/10d)')
ax.set_title('CST vs Mathematical Model: Heat Production Estimates', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('fig5_cst_vs_model_hp.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 5 saved.')

In [ ]:
# Scatter: CST vs Model agreement
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(hp_cst[:n], hp_model[:n], color='#2E86AB', s=90, zorder=3)

for i, d in enumerate(diets_cst[:n]):
    ax.annotate(d, (hp_cst[i], hp_model[i]),
                textcoords='offset points', xytext=(5, 3), fontsize=8)

lims = [min(hp_cst[:n].min(), hp_model[:n].min()) - 20,
        max(hp_cst[:n].max(), hp_model[:n].max()) + 20]
ax.plot(lims, lims, 'k--', linewidth=1, label='Line of equality')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('HP by CST (kcal/bird/10d)')
ax.set_ylabel('HP by Mathematical Model (kcal/bird/10d)')
ax.set_title('Agreement Between CST and Model: HP Estimates', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('fig6_cst_model_agreement.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figure 6 saved.')

---
## 9. Summary Statistics

In [ ]:
summary = means[means['Diet'].isin(all_diets_order)][[
    'Diet', 'MEI_mean', 'HP_mean', 'RE_mean', 'REF_mean', 'REP_mean', 'KREF_mean', 'KREP_mean'
]].copy()

summary['HP_pct_MEI'] = (summary['HP_mean'] / summary['MEI_mean'] * 100).round(1)
summary['RE_pct_MEI'] = (summary['RE_mean'] / summary['MEI_mean'] * 100).round(1)

summary.columns = ['Diet', 'MEI', 'HP', 'RE', 'REF', 'REP', 'KREF', 'KREP', 'HP%', 'RE%']
summary = summary.round(2)
print('Summary Statistics (per bird, 10-day pre-starter period)')
print(summary.to_string(index=False))

In [ ]:
print('\nKey Findings:')
top_mei = summary.loc[summary['MEI'].idxmax(), 'Diet']
low_hp  = summary.loc[summary['HP'].idxmin(), 'Diet']
top_kref = summary.loc[summary['KREF'].idxmax(), 'Diet']
top_krep = summary.loc[summary['KREP'].idxmax(), 'Diet']

print(f'  Highest MEI:   {top_mei} ({summary["MEI"].max():.1f} kcal/bird/10d)')
print(f'  Lowest HP:     {low_hp}  ({summary["HP"].min():.1f} kcal/bird/10d)')
print(f'  Highest KREF:  {top_kref} ({summary["KREF"].max():.4f})')
print(f'  Highest KREP:  {top_krep} ({summary["KREP"].max():.4f})')
print(f'  Mean HP as % of MEI: {summary["HP%"].mean():.1f}%')
print(f'  Mean RE as % of MEI: {summary["RE%"].mean():.1f}%')
print('\nConclusion: KREP > KREF across all diets confirms protein accretion is more energetically')
print('efficient than fat deposition in pre-starter broiler chicks.')
print('The mathematical model underestimates HP vs CST, likely due to its lower maintenance')
print('parameter sensitivity at this early growth stage.')

---
## 10. References

- Owolabi, S. J. (2025). *Comparison of Energy Partitioning Indices in Pre-starter Broiler Chicks.* MSc Thesis, University of Ibadan.
- van der Klein, S. A. S., et al. (2020). Mathematical modeling of energy partitioning in broilers. *Poultry Science.*
- Larbier, M., & Leclercq, B. (1992). *Nutrition and Feeding of Poultry.* Nottingham University Press.